In [1]:
!pip install numpy pandas matplotlib scikit-learn

In [3]:
"""
AI-Driven Intelligent Network for Smart Campus Applications

Models:
1. Naive Baseline
2. Linear Regression
3. Random Forest
4. Isolation Forest

Run:
pip install numpy pandas matplotlib scikit-learn
python smart_campus_ai.py
"""

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestRegressor, IsolationForest
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    precision_score,
    recall_score,
)

# --------------------------------------------------
# 1. SETTINGS
# --------------------------------------------------

RANDOM_STATE = 42
DAYS = 60
HOURS = DAYS * 24
TRAIN_RATIO = 0.75

OUTPUT_DIR = Path("results")
OUTPUT_DIR.mkdir(exist_ok=True)

# --------------------------------------------------
# 2. SIMULATE CAMPUS NETWORK TRAFFIC
# --------------------------------------------------

rng = np.random.default_rng(RANDOM_STATE)

timestamps = pd.date_range(
    start="2026-01-01",
    periods=HOURS,
    freq="h"
)

hour = timestamps.hour.to_numpy()
day_of_week = timestamps.dayofweek.to_numpy()

daily_user_pattern = (
    55 + 95 * np.exp(-((hour - 13) / 4.5) ** 2)
)

weekend_effect = np.where(day_of_week >= 5, -35, 0)

users = (
    daily_user_pattern
    + weekend_effect
    + rng.normal(0, 8, HOURS)
)

users = np.clip(users, 5, None).round().astype(int)

evening_load = 35 * np.exp(-((hour - 20) / 3.5) ** 2)

bandwidth = (
    2.2 * users
    + evening_load
    + 45
    + rng.normal(0, 18, HOURS)
)

bandwidth = np.clip(bandwidth, 10, None)

# Insert 24 artificial traffic spikes in the test period
split_point = int(HOURS * TRAIN_RATIO)

spike_positions = rng.choice(
    np.arange(split_point, HOURS),
    size=24,
    replace=False
)

bandwidth[spike_positions] += rng.uniform(
    180, 350, size=24
)

df = pd.DataFrame({
    "timestamp": timestamps,
    "hour": hour,
    "day_of_week": day_of_week,
    "users": users,
    "bandwidth_mbps": bandwidth,
})

df["is_anomaly"] = 0
df.loc[spike_positions, "is_anomaly"] = 1

# --------------------------------------------------
# 3. FEATURE ENGINEERING
# --------------------------------------------------

df["bandwidth_lag_1"] = df["bandwidth_mbps"].shift(1)
df["bandwidth_lag_24"] = df["bandwidth_mbps"].shift(24)
df["users_lag_1"] = df["users"].shift(1)

df = df.dropna().reset_index(drop=True)

features = [
    "hour",
    "day_of_week",
    "users",
    "bandwidth_lag_1",
    "bandwidth_lag_24",
    "users_lag_1",
]

X = df[features]
y = df["bandwidth_mbps"]

split_index = int(len(df) * TRAIN_RATIO)

X_train = X.iloc[:split_index]
X_test = X.iloc[split_index:]

y_train = y.iloc[:split_index]
y_test = y.iloc[split_index:]

test_df = df.iloc[split_index:].copy()

# --------------------------------------------------
# 4. NAIVE BASELINE
# --------------------------------------------------

naive_predictions = (
    test_df["bandwidth_lag_24"].to_numpy()
)

# --------------------------------------------------
# 5. LINEAR REGRESSION
# --------------------------------------------------

linear_model = LinearRegression()

linear_model.fit(X_train, y_train)

linear_predictions = linear_model.predict(X_test)

# --------------------------------------------------
# 6. RANDOM FOREST
# --------------------------------------------------

random_forest_model = RandomForestRegressor(
    n_estimators=150,
    max_depth=12,
    min_samples_split=4,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

random_forest_model.fit(X_train, y_train)

rf_predictions = random_forest_model.predict(X_test)

# --------------------------------------------------
# 7. MODEL EVALUATION
# --------------------------------------------------

def calculate_metrics(actual, predicted):

    return {
        "MAE (Mbps)": mean_absolute_error(
            actual, predicted
        ),

        "RMSE (Mbps)": np.sqrt(
            mean_squared_error(actual, predicted)
        ),

        "R2 Score": r2_score(
            actual, predicted
        )
    }


results = pd.DataFrame({
    "Naive Baseline": calculate_metrics(
        y_test, naive_predictions
    ),

    "Linear Regression": calculate_metrics(
        y_test, linear_predictions
    ),

    "Random Forest": calculate_metrics(
        y_test, rf_predictions
    )

}).T

print("\n========== BANDWIDTH PREDICTION RESULTS ==========")
print(results.round(3))

results.to_csv(
    OUTPUT_DIR / "model_comparison.csv"
)

# --------------------------------------------------
# 8. ISOLATION FOREST ANOMALY DETECTION
# --------------------------------------------------

anomaly_features = [
    "users",
    "bandwidth_mbps"
]

X_anomaly_train = df.iloc[:split_index][
    anomaly_features
]

X_anomaly_test = test_df[
    anomaly_features
]

isolation_model = IsolationForest(
    n_estimators=150,
    contamination=0.05,
    random_state=RANDOM_STATE
)

isolation_model.fit(X_anomaly_train)

raw_predictions = isolation_model.predict(
    X_anomaly_test
)

# -1 = anomaly, 1 = normal
anomaly_predictions = np.where(
    raw_predictions == -1, 1, 0
)

actual_anomalies = test_df[
    "is_anomaly"
].to_numpy()

precision = precision_score(
    actual_anomalies,
    anomaly_predictions,
    zero_division=0
)

recall = recall_score(
    actual_anomalies,
    anomaly_predictions,
    zero_division=0
)

print("\n========== ANOMALY DETECTION RESULTS ==========")

print(
    "Actual anomalies:",
    int(actual_anomalies.sum())
)

print(
    "Detected anomalies:",
    int(anomaly_predictions.sum())
)

print("Precision:", round(precision, 3))
print("Recall:", round(recall, 3))

pd.DataFrame({
    "Metric": ["Precision", "Recall"],
    "Value": [precision, recall]
}).to_csv(
    OUTPUT_DIR / "anomaly_metrics.csv",
    index=False
)

# --------------------------------------------------
# 9. ACTUAL VS PREDICTED GRAPH
# --------------------------------------------------

plt.figure(figsize=(12, 5))

hours_to_plot = min(96, len(y_test))

plt.plot(
    test_df["timestamp"].iloc[:hours_to_plot],
    y_test.iloc[:hours_to_plot],
    label="Actual Bandwidth",
    linewidth=2
)

plt.plot(
    test_df["timestamp"].iloc[:hours_to_plot],
    rf_predictions[:hours_to_plot],
    label="Random Forest Prediction",
    linestyle="--"
)

plt.title("Actual vs Predicted Campus Network Bandwidth")
plt.xlabel("Time")
plt.ylabel("Bandwidth (Mbps)")
plt.legend()
plt.xticks(rotation=30)
plt.tight_layout()

plt.savefig(
    OUTPUT_DIR / "actual_vs_predicted.png",
    dpi=300
)

plt.close()

# --------------------------------------------------
# 10. FEATURE IMPORTANCE GRAPH
# --------------------------------------------------

importance = pd.Series(
    random_forest_model.feature_importances_,
    index=features
).sort_values()

plt.figure(figsize=(8, 5))

importance.plot(kind="barh")

plt.title("Random Forest Feature Importance")
plt.xlabel("Importance")
plt.ylabel("Features")
plt.tight_layout()

plt.savefig(
    OUTPUT_DIR / "feature_importance.png",
    dpi=300
)

plt.close()

# --------------------------------------------------
# 11. ANOMALY DETECTION GRAPH
# --------------------------------------------------

plt.figure(figsize=(12, 5))

normal_mask = anomaly_predictions == 0
anomaly_mask = anomaly_predictions == 1

plt.scatter(
    test_df.loc[normal_mask, "timestamp"],
    test_df.loc[normal_mask, "bandwidth_mbps"],
    label="Normal Traffic",
    s=18
)

plt.scatter(
    test_df.loc[anomaly_mask, "timestamp"],
    test_df.loc[anomaly_mask, "bandwidth_mbps"],
    label="Detected Anomaly",
    marker="x",
    s=55
)

plt.title("Isolation Forest Anomaly Detection")
plt.xlabel("Time")
plt.ylabel("Bandwidth (Mbps)")
plt.legend()
plt.xticks(rotation=30)
plt.tight_layout()

plt.savefig(
    OUTPUT_DIR / "anomaly_detection.png",
    dpi=300
)

plt.close()

# --------------------------------------------------
# 12. SAVE TEST PREDICTIONS
# --------------------------------------------------

prediction_output = test_df[
    [
        "timestamp",
        "users",
        "bandwidth_mbps",
        "is_anomaly"
    ]
].copy()

prediction_output["naive_prediction"] = (
    naive_predictions
)

prediction_output["linear_regression_prediction"] = (
    linear_predictions
)

prediction_output["random_forest_prediction"] = (
    rf_predictions
)

prediction_output["isolation_forest_flag"] = (
    anomaly_predictions
)

prediction_output.to_csv(
    OUTPUT_DIR / "test_predictions.csv",
    index=False
)

# --------------------------------------------------
# 13. FINAL OUTPUT
# --------------------------------------------------

print("\nAll results generated successfully!")

print("\nFiles saved in:", OUTPUT_DIR.resolve())

for file in sorted(OUTPUT_DIR.iterdir()):
    print("-", file.name)


========== BANDWIDTH PREDICTION RESULTS ==========
                   MAE (Mbps)  RMSE (Mbps)  R2 Score
Naive Baseline         67.148      103.974     0.094
Linear Regression      32.770       74.076     0.540
Random Forest          33.733       74.447     0.535

========== ANOMALY DETECTION RESULTS ==========
Actual anomalies: 24
Detected anomalies: 37
Precision: 0.649
Recall: 1.0

All results generated successfully!

Files saved in: /content/results
- actual_vs_predicted.png
- anomaly_detection.png
- anomaly_metrics.csv
- feature_importance.png
- model_comparison.csv
- test_predictions.csv


In [4]:
import shutil
from google.colab import files

shutil.make_archive("smart_campus_results", "zip", "results")
files.download("smart_campus_results.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>